# zagg write pipeline

Lambda fan-out from the shard maps built in `01_query`: dispatch, futures, cost. Same shardmap, different aggregations; with and without the strict-AOI mask; California at o8 with cost telemetry.

In [1]:
# %pip install "zagg[catalog,viz]"

import io
import json
import logging
import os
import re
import time
from importlib import resources
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import Markdown

from zagg.catalog import load_polygon
from zagg.catalog.shardmap import ShardMap
from zagg.catalog.sources import Catalog
from zagg.client import Run
from zagg.config import default_config
from zagg.data import demo_aoi
from zagg.grids import from_config
from zagg import notebook as znb
from zagg.notebook import format_max_cost, max_cost_preview

In [2]:
os.environ.setdefault("AWS_PROFILE", "nasa")
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
for noisy in ("botocore", "boto3", "urllib3", "s3transfer", "stac_geoparquet"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

OUT = Path("outputs")
STORE = "s3://sliderule-public/zagg-demo"

timings = {}


class stage:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        timings[self.name] = round(time.perf_counter() - self.t0, 2)
        print(f"[{self.name}] {timings[self.name]:.1f}s")

## The aggregation template, block by block

In [3]:
raw = (resources.files("zagg.configs") / "atl03_tdigest_healpix_hive.yaml").read_text()


def yaml_block(text, name):
    block = re.search(rf"^{name}:.*?(?=^\w|\Z)", text, re.S | re.M).group(0)
    clean = "\n".join(l for l in block.splitlines() if not l.strip().startswith("#"))
    return Markdown(f"```yaml\n{clean}\n```")

### `data_source` — what to read

- h5coro reads each ATL03 granule directly over S3 by byte range — nothing is downloaded; six beam groups each contribute photon lat/lon/height.
- `index: {backend: inline}` selects the virtual chunk-index for the byte-range reads (vs the pure `hierarchical` metadata walk).
- `filters` drops TEP returns; `levels` + `read_plan` declare the segment→photon index link, so a worker fetches only the segments crossing its shard (`pad: 1`).

In [4]:
yaml_block(raw, "data_source")

```yaml
data_source:
  granule_workers: 4
  reader: h5coro
  driver: s3
  index:
    backend: inline            # virtual chunk-index for the byte-range reads;
  groups: [gt1l, gt1r, gt2l, gt2r, gt3l, gt3r]
  coordinates:
    latitude: "/{group}/heights/lat_ph"
    longitude: "/{group}/heights/lon_ph"
  variables:
    h_ph: "/{group}/heights/h_ph"
  filters:
    - dataset: "/{group}/heights/signal_conf_ph"
      column: 0                  # land surface type; TEP is uniform across columns
      op: ne
      value: -2
  base_level: photons
  levels:
    photons:
      path: "/{group}/heights"
      coordinates: {latitude: lat_ph, longitude: lon_ph}
      link: null
    segments:
      path: "/{group}/geolocation"
      coordinates: {latitude: reference_photon_lat, longitude: reference_photon_lon}
      link:
        to: photons
        index_beg: "/{group}/geolocation/ph_index_beg"
        count: "/{group}/geolocation/segment_ph_cnt"
        index_base: 1            # ATL03 ph_index_beg is 1-based per the v3 dict
  read_plan:
    spatial_index: segments
    pad: 1                       # one segment of padding on each side per #43

```

### `aggregation` — what to compute

- One output field per entry: `count` is a dense per-cell int32; `h_tdigest` is a ragged per-cell t-digest sketch (`delta: 256` centroid budget) from which any quantile is recovered at read time.
- `streaming: {mode: spill}` bounds worker memory: reads spill to `/tmp` partitions and aggregate in one pass after the reads (pooled-identical in the single-block regime; `merge` is the incremental-fold alternative for mergeable reducers).
- Swapping this block changes the science without touching the read or the layout — demoed below with the located variant.

In [5]:
yaml_block(raw, "aggregation")

```yaml
aggregation:
  streaming:
    mode: spill                # bounded read buffers spilled to /tmp partitions, one
  coordinates:
    morton:
      dtype: uint64
      fill_value: 0
  variables:
    count:
      function: len
      source: h_ph
      dtype: int32
      fill_value: 0
    h_tdigest:
      kind: ragged
      function: "zagg.stats.tdigest.build_tdigest"
      source: h_ph
      inner_shape: [2]
      params:
        delta: 256               # centroid budget (accuracy knob)
      dtype: float32
      fill_value: 0

```

### `output` — where it lands

- HEALPix nested grid: `child_order: 19` (~10 m cells); `parent_order: 9` is the dispatch shard (1024×1024 cells, one Lambda invoke each); `chunk_inner: 13` bundles 256 inner chunks into one sharded zarr object.
- `store_layout: hive`: every shard writes its own self-describing leaf zarr; `pyramid: false` keeps the overview sweep off.

In [6]:
yaml_block(raw, "output")

```yaml
output:
  grid:
    type: healpix
    indexing_scheme: nested
    parent_order: 9              # shard / dispatch unit: 4^(19-9) = 1024x1024 cells
    chunk_inner: 13              # inner Zarr chunk: 4^(19-13) = 64x64 cells (K=256 per shard)
    sharded: true                # one ShardingCodec object per array per leaf
    child_order: 19              # leaf cell resolution (~10 m)
  store_layout: hive             # each dispatched shard writes its own leaf zarr
  pyramid: false                 # overview sweep off (untested at demo scale, issue #201)

```

### `worker` — which Lambda runs it

- Selects the pre-provisioned variant: `memory` (2048/4096/8192 MB → `process-shard-<memory>`), `extra_disk: true` picks the `-disk` twin (memory+2048 MB of `/tmp`), which backs the spill partitions.
- The pre-invoke cost ceiling and the metered post-run rollup both key off this block.

In [7]:
yaml_block(raw, "worker")

```yaml
worker:
  memory: 4096                   # pre-provisioned Lambda variant, MB (2048|4096|8192) --
  extra_disk: true               # the -disk twin: memory+2048 MB of /tmp, backing the
```

## SERC — four variants, dispatched in parallel

Each `dispatch()` returns after its setup handshake and `progress_async()` keeps the kernel free, so all four fleets launch back-to-back; one join below blocks once, then the summary compares them.

In [8]:
def clear_store(prefix):
    """Delete every object under an s3://bucket/prefix (paged batch deletes)."""
    bucket, _, key = prefix.removeprefix("s3://").partition("/")
    s3 = boto3.client("s3")
    n = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=key):
        objs = [{"Key": o["Key"]} for o in page.get("Contents", [])]
        for i in range(0, len(objs), 1000):
            s3.delete_objects(Bucket=bucket, Delete={"Objects": objs[i : i + 1000], "Quiet": True})
        n += len(objs)
    return n


# SERC demo stores are wiped for a clean re-run (the hive template guard refuses
# re-templating over a populated digit tree); California below is KEPT, never wiped.
for name in ("serc_tdigest", "serc_tdigest_aoi", "serc_tdigest_located", "serc_tdigest_strata", "s2_serc_2025"):
    print(f"{name}.zarr: {clear_store(f'{STORE}/{name}.zarr')} objects cleared")

serc_tdigest.zarr: 96 objects cleared
serc_tdigest_aoi.zarr: 104 objects cleared
serc_tdigest_located.zarr: 104 objects cleared
serc_tdigest_strata.zarr: 128 objects cleared
s2_serc_2025.zarr: 10 objects cleared


In [9]:
serc_map = str(OUT / "shardmap_atl03_serc_o9.json")
config = default_config("atl03_tdigest_healpix_hive")

run = Run.from_config(config, shardmap=serc_map, store=f"{STORE}/serc_tdigest.zarr", overwrite=True)
run

Run(4 shards -> 's3://sliderule-public/zagg-demo/serc_tdigest.zarr', function='process-shard-4096-disk', region='us-west-2')

In [10]:
t0_run = time.perf_counter()
handle = run.dispatch()  # returns after the setup handshake; every shard is a Future
handle.progress_async()  # live bar on a daemon thread -- the kernel is NOT blocked
handle.status()

zagg.runner: Account concurrency: limit=2000, current=0, padding=100, available=1900 -> using 4 workers
zagg.client: Max cost ceiling: ~$0.19 (4 units x 4 GB x 900s, arm64)
earthaccess.auth: You're now authenticated with NASA Earthdata Login
earthaccess.auth: Using token with expiration date 08/27/2026
zagg.runner: Preflight OK (function zagg version 0.0.0+unknown)


{'pending': 4, 'ok': 0, 'failed': 0}

shards:   0%|          | 0/4 [00:00<?, ?shard/s]

In [21]:
handle.status() | {"cost_so_far_usd": round(handle.cost_usd(), 4)}  # rerun at will while the fleet works

{'pending': 3, 'ok': 1, 'failed': 0, 'cost_so_far_usd': 0.0041}

## SERC — same pipeline, with the strict-AOI mask

In [12]:
cat_serc = Catalog.from_geoparquet(str(OUT / "catalog_atl03_serc.parquet"))
serc_parts = load_polygon(demo_aoi("serc"))

masked_config = default_config("atl03_tdigest_healpix_hive")
masked_config.output["aoi_mask"] = True
grid = from_config(masked_config)

sm_masked = ShardMap.build(cat_serc, grid, region=serc_parts, mortie_order=9)
sm_masked.to_json(str(OUT / "shardmap_atl03_serc_o9_aoi.json"))

# in-AOI cell fraction per shard: edge shards are mostly outside the flight box
{
    grid.shard_label(int(k)): round(float(grid.aoi_mask_from_payload(m, grid.children(int(k))).mean()), 3)
    for k, m in zip(sm_masked.shard_keys, sm_masked.aoi_mask)
}

{'4331422411': 0.003,
 '4331422412': 0.566,
 '4331422414': 0.045,
 '4331422421': 0.006}

In [13]:
run_masked = Run.from_config(
    masked_config,
    shardmap=str(OUT / "shardmap_atl03_serc_o9_aoi.json"),
    store=f"{STORE}/serc_tdigest_aoi.zarr",
    overwrite=True,
)
handle_masked = run_masked.dispatch()
handle_masked.progress_async()
handle_masked.status()

zagg.runner: Account concurrency: limit=2000, current=0, padding=100, available=1900 -> using 4 workers
zagg.client: Max cost ceiling: ~$0.19 (4 units x 4 GB x 900s, arm64)
zagg.runner: Preflight OK (function zagg version 0.0.0+unknown)


shards:   0%|          | 0/4 [00:00<?, ?shard/s]

{'pending': 4, 'ok': 0, 'failed': 0}

In [22]:
handle_masked.status() | {"cost_so_far_usd": round(handle_masked.cost_usd(), 4)}  # rerun at will while the fleet works

{'pending': 3, 'ok': 1, 'failed': 0, 'cost_so_far_usd': 0.0038}

zagg.runner: Dispatched run stats write (4 rows, fire-and-forget): s3://sliderule-public/zagg-demo/serc_tdigest.zarr/stats_20260803T230339Z_45360d7566144e8ab0168c4e4bc178e2.parquet
zagg.runner: Dispatched run stats write (4 rows, fire-and-forget): s3://sliderule-public/zagg-demo/serc_tdigest_aoi.zarr/stats_20260803T230410Z_1025bf61bbe74ddb978e6e4463e7e20d.parquet
zagg.runner: Dispatched run stats write (4 rows, fire-and-forget): s3://sliderule-public/zagg-demo/serc_tdigest_located.zarr/stats_20260803T230414Z_cc01f1372b6e455e89128c6dde055fc2.parquet
zagg.runner: Dispatched run stats write (4 rows, fire-and-forget): s3://sliderule-public/zagg-demo/serc_tdigest_strata.zarr/stats_20260803T230452Z_aaa8f256c2684a1a83e1b790c5976f2f.parquet


The mask drops no photons — the masked store additionally carries a per-cell `aoi_mask` bool array, which readers use to clip whole-shard overhang at the AOI edge.

## SERC — same shardmap, different aggregation: located t-digest

Same `build_tdigest` reducer — the single delta is `location: leaf_id`, which adds a per-centroid morton location channel (order-29 photon words folded via `common_ancestor`, stored as the row-aligned `h_tdigest_locations` companion array).

In [15]:
located_config = default_config("atl03_tdigest_healpix_hive")
# graft the located VARIABLES only -- coordinates and the streaming block stay the template's
located_config.aggregation["variables"] = default_config("atl03_tdigest_located_healpix").aggregation["variables"]

located_yaml = (resources.files("zagg.configs") / "atl03_tdigest_located_healpix.yaml").read_text()
yaml_block(located_yaml, "aggregation")

```yaml
aggregation:
  coordinates:
    morton:
      dtype: uint64
      fill_value: 0
  variables:
    count:
      function: len
      source: h_ph
      dtype: int32
      fill_value: 0
    h_tdigest:
      kind: ragged
      function: "zagg.stats.tdigest.build_tdigest"
      source: h_ph
      location: leaf_id
      inner_shape: [2]
      params:
        delta: 256
      dtype: float32
      fill_value: 0

```

In [16]:
run_located = Run.from_config(
    located_config, shardmap=serc_map, store=f"{STORE}/serc_tdigest_located.zarr", overwrite=True
)
handle_located = run_located.dispatch()
handle_located.progress_async()
handle_located.status()

zagg.runner: Account concurrency: limit=2000, current=0, padding=100, available=1900 -> using 4 workers
zagg.client: Max cost ceiling: ~$0.19 (4 units x 4 GB x 900s, arm64)
zagg.runner: Preflight OK (function zagg version 0.0.0+unknown)


shards:   0%|          | 0/4 [00:00<?, ?shard/s]

{'pending': 4, 'ok': 0, 'failed': 0}

## SERC — signal/noise strata (the split t-digest)

The production-direction template: TWO digests per cell — `h_tdigest_signal` / `h_tdigest_noise`, split by the per-surface confidence union (`>= 2` on any of the five `signal_conf_ph` columns, declared as column-selected variables in `data_source`) — plus a packed per-cell `composition` lanes field. Both strata also carry `location: leaf_id`, so strata and location compose.

In [17]:
strata_config = default_config("atl03_tdigest_strata_healpix")
strata_config.output = default_config("atl03_tdigest_healpix_hive").output  # same hive grid/layout
strata_config.worker = dict(default_config("atl03_tdigest_healpix_hive").worker)  # 4096-disk
strata_config.aggregation["streaming"] = {"mode": "spill"}  # same spill path as the template

strata_yaml = (resources.files("zagg.configs") / "atl03_tdigest_strata_healpix.yaml").read_text()
yaml_block(strata_yaml, "aggregation")

```yaml
aggregation:
  coordinates:
    morton:
      dtype: uint64
      fill_value: 0
  variables:
    count:
      function: len
      source: h_ph
      dtype: int32
      fill_value: 0
    h_tdigest_signal:
      kind: ragged
      function: "zagg.stats.tdigest.build_tdigest_where"
      source: h_ph
      location: leaf_id
      inner_shape: [2]
      params:
        delta: 256
        where: >-
          (signal_conf_land >= 2) | (signal_conf_ocean >= 2) |
          (signal_conf_sea_ice >= 2) | (signal_conf_land_ice >= 2) |
          (signal_conf_inland_water >= 2)
      dtype: float32
      fill_value: 0
      attrs:
        stratum: signal
        signal_threshold: 2
    h_tdigest_noise:
      kind: ragged
      function: "zagg.stats.tdigest.build_tdigest_where"
      source: h_ph
      location: leaf_id
      inner_shape: [2]
      params:
        delta: 256
        where: >-
          ~((signal_conf_land >= 2) | (signal_conf_ocean >= 2) |
          (signal_conf_sea_ice >= 2) | (signal_conf_land_ice >= 2) |
          (signal_conf_inland_water >= 2))
      dtype: float32
      fill_value: 0
      attrs:
        stratum: noise
        signal_threshold: 2
    composition:
      function: "zagg.stats.composition.pack_composition"
      source: h_ph
      dtype: uint64
      fill_value: 0
      params:
        conf_land: signal_conf_land
        conf_ocean: signal_conf_ocean
        conf_sea_ice: signal_conf_sea_ice
        conf_land_ice: signal_conf_land_ice
        conf_inland_water: signal_conf_inland_water
        threshold: 2
      attrs:
        composition:
          spec: zagg-composition/1
          lanes: [land, ocean, sea_ice, land_ice, inland_water, low, med, high]
          of: h_tdigest_signal        # N_signal = this digest's total weight
          threshold: 2

```

In [18]:
run_strata = Run.from_config(
    strata_config, shardmap=serc_map, store=f"{STORE}/serc_tdigest_strata.zarr", overwrite=True
)
handle_strata = run_strata.dispatch()
handle_strata.progress_async()
handle_strata.status()

zagg.runner: Account concurrency: limit=2000, current=0, padding=100, available=1900 -> using 4 workers
zagg.client: Max cost ceiling: ~$0.19 (4 units x 4 GB x 900s, arm64)
zagg.runner: Preflight OK (function zagg version 0.0.0+unknown)


shards:   0%|          | 0/4 [00:00<?, ?shard/s]

{'pending': 4, 'ok': 0, 'failed': 0}

All four fleets are in flight — join once, then compare:

In [23]:
for h in (handle, handle_masked, handle_located, handle_strata):
    h.wait()
timings["SERC: 4 variants (parallel)"] = round(time.perf_counter() - t0_run, 2)
print(f"all four runs complete in {timings['SERC: 4 variants (parallel)']:.1f}s wall (parallel)")
{
    name: h.status()
    for name, h in [
        ("combined", handle),
        ("aoi_mask", handle_masked),
        ("located", handle_located),
        ("strata", handle_strata),
    ]
}

all four runs complete in 270.8s wall (parallel)


{'combined': {'pending': 0, 'ok': 4, 'failed': 0},
 'aoi_mask': {'pending': 0, 'ok': 4, 'failed': 0},
 'located': {'pending': 0, 'ok': 4, 'failed': 0},
 'strata': {'pending': 0, 'ok': 4, 'failed': 0}}

In [24]:
res = next(iter(handle.futures.values())).result()
print(f"metered cost: ${handle.cost_usd():.4f} (billed-duration rollup, same figure the CLI prints)")
{k: res.get(k) for k in ("shard_key", "wall_time", "lambda_duration", "retries")} | {
    k: v for k, v in res["body"].items() if isinstance(v, (int, float, str))
}

metered cost: $0.0213 (billed-duration rollup, same figure the CLI prints)


{'shard_key': 5347395636851376137,
 'wall_time': 106.11999416351318,
 'lambda_duration': 101.634794,
 'retries': 0,
 'cells_with_data': 104941,
 'total_obs': 3741239,
 'granule_count': 68,
 'files_processed': 68,
 'duration_s': 101.634794,
 'container_hwm_mb': 555.15625,
 'max_memory_mb': 546.96875,
 'cpu_seconds': 115.97,
 'container_cold': False,
 'container_generation': 3,
 'rss_start_mb': 127.984375,
 'sandbox_id': '2026/08/03/[$LATEST]962bcbf660c64f9b86a46ecfa4cac778',
 'container_init_ts': 1785798090.010791}

Cost across the four SERC variants — same shardmap, same photons, different aggregation blocks:

In [25]:
def rollup(h):
    bodies = [f.result().get("body", {}) for f in h.futures.values()]
    return {
        "cells_with_data": sum(b.get("cells_with_data") or 0 for b in bodies),
        "total_obs": sum(b.get("total_obs") or 0 for b in bodies),
    }


pd.DataFrame(
    {
        "combined": {"cost_usd": handle.cost_usd(), **rollup(handle)},
        "combined + aoi_mask": {"cost_usd": handle_masked.cost_usd(), **rollup(handle_masked)},
        "located": {"cost_usd": handle_located.cost_usd(), **rollup(handle_located)},
        "signal/noise strata (located)": {"cost_usd": handle_strata.cost_usd(), **rollup(handle_strata)},
    }
).T.round(4)

,cost_usd,cells_with_data,total_obs
combined,0.0213,417322.0,20936129.0
combined + aoi_mask,0.0217,417322.0,20936129.0
located,0.0211,417322.0,20936129.0
signal/noise strata (located),0.0282,417322.0,20936129.0


## SERC — Sentinel-2 raster write (the raster module)

- Same grid family: each datatake's COGs are sampled at the o19 cell centres into `(time, cells)` arrays — the S2 store shares the cells axis with the photon stores, so co-registration is index arithmetic.
- `zagg.client` v1 is point-path-only; raster dispatch rides `zagg.notebook.run` (the `agg` wrapper: ceiling print, progress bar, `RunView` cost report).
- One year (2025) to bound the demo — drop the year filter to ingest the full archive.

In [26]:
s2_yaml = (resources.files("zagg.configs") / "sentinel2_l2a.yaml").read_text()
# raster templates have no aggregation block: the reader IS the aggregation
# (each band pull-sampled at the o19 cell centres into (time, cells) arrays)
Markdown("\n".join(yaml_block(s2_yaml, block).data for block in ("data_source", "output")))

```yaml
data_source:
  reader: raster
  bands:
    red: {asset: red, dtype: uint16, fill_value: 0, scale: 0.0001, offset: -0.1}
    green: {asset: green, dtype: uint16, fill_value: 0, scale: 0.0001, offset: -0.1}
    blue: {asset: blue, dtype: uint16, fill_value: 0, scale: 0.0001, offset: -0.1}
    nir: {asset: nir, dtype: uint16, fill_value: 0, scale: 0.0001, offset: -0.1}
    scl: {asset: scl, dtype: uint8, fill_value: 0}
  nodata: 0
  anonymous: true

```
```yaml
output:
  grid:
    type: healpix
    parent_order: 11
    child_order: 19
  store: ./sentinel2_l2a.zarr
```

In [27]:
s2_full = Catalog.from_geoparquet(str(OUT / "catalog_s2_serc.parquet"))
s2_2025 = Catalog(
    s2_full.table.filter(pc.equal(pc.year(s2_full.table.column("datetime")), 2025)),
    dict(s2_full.metadata),
)

s2_config = default_config("sentinel2_l2a")
s2_config.output["grid"]["parent_order"] = 9

sm_s2_2025 = ShardMap.build(s2_2025, from_config(s2_config), region=serc_parts, mortie_order=9)
sm_s2_2025.to_json(str(OUT / "shardmap_s2_serc_2025_o9.json"))
print(
    f"{len(s2_2025):,} items in 2025 -> {sm_s2_2025.metadata['total_shards']} shards, "
    f"{sm_s2_2025.metadata['total_pairs']:,} (shard, datatake) pairs"
)

85 items in 2025 -> 4 shards, 340 (shard, datatake) pairs


In [28]:
with stage("SERC: S2 raster run (2025)"):
    s2_view = znb.run(
        s2_config,
        catalog=str(OUT / "shardmap_s2_serc_2025_o9.json"),
        backend="lambda",
        store=f"{STORE}/s2_serc_2025.zarr",
        overwrite=True,
        profile=True,  # per-stage fetch/decode/sample/write splits in the stats parquet
    )
s2_view

Max cost ceiling: ~$0.19 (4 units x 4 GB x 900s, arm64)


shards:   0%|          | 0/4 [00:00<?, ?unit/s]

zagg.runner: Max cost ceiling: ~$0.19 (4 units x 4 GB x 900s, arm64)
zagg.runner: Async raster results at s3://sliderule-public/zagg-demo/s2_serc_2025.zarr.status/11adf258ba614388982978b519b9ad5b
zagg.runner: Preflight OK (function zagg version 0.0.0+unknown)
zagg.runner: Dispatched run stats write (4 rows, fire-and-forget): s3://sliderule-public/zagg-demo/s2_serc_2025.zarr/stats_20260803T231517Z_11adf258ba614388982978b519b9ad5b.parquet
zagg.runner: Dispatched rollup sweep (4 leaves, fire-and-forget)
zagg.runner: Done (lambda): 4/4 shards, 0 errors, 514.1s


[SERC: S2 raster run (2025)] 524.5s


cost,USD
max (pre-invoke ceiling),$0.1920
estimated (prior runs),n/a
actual (billed-duration rollup),$0.1005
run,
units,4
with data,4
errors,0
observations,340
wall time (s),514.1
lambda time (s),"1,885.0"


## California at o8 — cost ceiling, first-shard latency, actuals

The full production shape at state scale: the **strata (signal/noise, located)** aggregation — two `build_tdigest_where` digests split by the per-surface confidence union, each with a morton location channel, plus the packed `composition` lanes — on hive + sharded o8 shards, `process-shard-4096-disk` with spill, and the **sidecar** chunk-index in write-through mode (`on_miss: build`). The store is **kept**, never wiped — rerun to a fresh store path and compare wall/cost against the reported run to see the warm-index difference. (Hive leaves also write D20 stats sidecars — telemetry, unrelated to the chunk-index.)

In [ ]:
ca_config = default_config("atl03_tdigest_strata_healpix")
ca_config.output = default_config("atl03_tdigest_healpix_hive").output  # hive + sharded + pyramid off
ca_config.output["grid"]["parent_order"] = 8
ca_config.worker = dict(default_config("atl03_tdigest_healpix_hive").worker)  # 4096-disk
ca_config.aggregation["streaming"] = {"mode": "spill"}
# write-through sidecar chunk-index: this run stores a per-granule index manifest
# as it reads; a repeat run over the same granules skips the metadata walk
ca_config.data_source["index"] = {
    "backend": "sidecar",
    "on_miss": "build",
    "store": "s3://sliderule-public-cors/zagg-index/ATL03/007",
}
ca_map = str(OUT / "shardmap_california_o8.json")

print(format_max_cost(max_cost_preview(ca_config, catalog=ca_map)))

In [ ]:
# KEPT across sessions (never wiped). Repeat runs: point at a fresh path -- the
# reported store stays, and the warmed sidecar index speeds the repeat either way.
CA_STORE = f"{STORE}/california_tdigest_o8.zarr"

run_ca = Run.from_config(ca_config, shardmap=ca_map, store=CA_STORE, overwrite=True)

t0 = time.perf_counter()
handle_ca = run_ca.dispatch()
completions = []
with stage("California: o8 run"):
    for fut in handle_ca.progress():
        completions.append(time.perf_counter() - t0)
print(
    f"first shard done at {completions[0]:.1f}s; "
    f"full state at {completions[-1]:.1f}s ({len(completions)} shards)"
)

Per-shard telemetry lands as a parquet at the store root (one row per shard: durations, phase timings, observation counts, priced GB-seconds).

In [ ]:
def fetch_run_stats(store_path):
    bucket, _, prefix = store_path.removeprefix("s3://").partition("/")
    s3 = boto3.client("s3")
    objs = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/stats_")["Contents"]
    key = max(o["Key"] for o in objs)  # timestamp-first names sort chronologically
    return pq.read_table(io.BytesIO(s3.get_object(Bucket=bucket, Key=key)["Body"].read())).to_pandas()


stats = fetch_run_stats(handle_ca.store_path)
print(
    f"actual cost ${stats.est_cost_usd.sum():.2f} across {len(stats)} shards; "
    f"{stats.duration_s.sum():,.0f} lambda-seconds, {stats.n_obs.sum() / 1e9:.2f}B photons"
)
stats[["duration_s", "n_obs", "n_granules", "gb_seconds", "est_cost_usd"]].describe().round(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

t = np.asarray(completions)
n = np.arange(1, len(t) + 1)
ax1.step(t, n, where="post", color="#4269d0", lw=2)
ax1.axvline(t[0], color="#9498a0", lw=1, ls="--")
ax1.annotate(f"first shard {t[0]:.0f}s", (t[0], len(t)), xytext=(6, -2), textcoords="offset points", fontsize=9, color="#555")
ax1.annotate(f"wall {t[-1]:.0f}s", (t[-1], len(t) * 0.6), xytext=(-6, 0), textcoords="offset points", ha="right", fontsize=9, color="#555")
ax1.set_xlabel("seconds since dispatch")
ax1.set_ylabel("shards complete")
ax1.set_title("Fleet completion — California o8", fontsize=11)

ax2.scatter(stats.n_obs / 1e6, stats.est_cost_usd * 100, s=12, alpha=0.45, color="#4269d0", edgecolors="none")
ax2.set_xlabel("photons aggregated (millions)")
ax2.set_ylabel("shard cost (¢)")
ax2.set_title("Per-shard cost vs photon volume", fontsize=11)

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.25, lw=0.5)
plt.tight_layout()

Per-shard distributions and the run rollup — including how many shards entered the spill fold regime (`spill_blocks_closed`, issue #370).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
panels = [("duration_s", "shard duration (s)", 1.0),
          ("est_cost_usd", "shard cost (\u00a2)", 100.0),
          ("n_obs", "observations (millions)", 1e-6)]
for ax, (col, label, scale) in zip(axes, panels):
    ax.hist(stats[col].dropna() * scale, bins=40, color="#4269d0", alpha=0.85)
    ax.set_xlabel(label)
    ax.set_ylabel("shards")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.25, lw=0.5)
plt.tight_layout()

In [ ]:
folded = stats.get("spill_blocks_closed")
run_summary = {
    "shards": len(stats),
    "succeeded": int(stats.success.sum()),
    "total_obs": f"{int(stats.n_obs.sum()):,}",
    "granule reads": f"{int(stats.n_granules.sum()):,}",
    "lambda-seconds": f"{stats.duration_s.sum():,.0f}",
    "actual cost (USD)": round(float(stats.est_cost_usd.sum()), 2),
    "duration s (median / p95 / max)": f"{stats.duration_s.median():.0f} / {stats.duration_s.quantile(0.95):.0f} / {stats.duration_s.max():.0f}",
    "obs per shard (median / max)": f"{stats.n_obs.median():,.0f} / {stats.n_obs.max():,.0f}",
    "spill fold regime (shards)": int((folded.fillna(0) > 0).sum()) if folded is not None else "n/a (pre-#370 worker)",
    "wall s (first / last shard)": f"{completions[0]:.0f} / {completions[-1]:.0f}",
}
pd.Series(run_summary, name="California o8").to_frame()

## Timings

In [ ]:
(OUT / "timings_write.json").write_text(json.dumps(timings, indent=2))
pd.Series(timings, name="seconds").to_frame()